In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.format('csv').option('inferSchema', True).option('header', True).load('/Volumes/workspace/default/pyspark_data/BigMart Sales.csv')

In [0]:
df.limit(10).display()

### SELECT

In [0]:
df.select('Item_Identifier', 'Item_Weight', 'Item_Fat_Content').limit(10).display()

In [0]:
df.select(col('Item_Identifier'), col('Item_Weight'), col('Item_Fat_Content')).limit(10).display()

### ALIAS

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).limit(10).display()

## FILTER


### Scenario - 1 Filter the data with fat content = Regular

In [0]:
df.filter(col('Item_Fat_Content')== 'Regular').limit(10).display()

### Scenario - 2 Slice the data with item type = Soft Drinks and weight < 10

In [0]:
df.filter((col('Item_Weight')<10) & (col('Item_Type')=='Soft Drinks')).limit(10).display()

### Scenario - 3 Fetch the data with Tier in (Tier 1 or Tier 2) and Outlet Size is Null

In [0]:
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1', 'Tier 2'))).limit(10).display()

### withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight', 'Item_Wt').limit(10).display()

### withColumn

#### Scenario 1

In [0]:
df.withColumn('flag', lit('new')).limit(10).display() # use lit function to add a constant value

#### Scenario 2

In [0]:
df.withColumn('MRP Per gm', col('Item_Weight')/col('Item_MRP')).limit(10).display()

#### Scenario 3

In [0]:
df.withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), 'Low Fat', 'LF'))\
    .withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), 'Regular', 'Reg')).limit(10).display()


### Type Casting

In [0]:
df  = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

### Sort/Order By

#### Scenario - 1

In [0]:
df.sort(col('Item_Weight').desc()).limit(10).display()

#### Scenario - 2

In [0]:
df.sort(col('Item_Visibility').asc()).limit(10).display() #ascending is default but its a good practice to write 

#### Scenario - 3

In [0]:
df.sort(['Item_Weight', 'Item_Visibility'], ascending=[False, True]).limit(10).display()

#### Scenario - 4

In [0]:
df.sort(['Item_Weight', 'Item_Visibility'], ascending=[0, 1]).limit(10).display() #0 is false and 1 is true

### Drop

#### Scenario - 1 (Dropping Single Column)

In [0]:
df.drop(col('Item_Visibility')).limit(10).display()

#### Scenario - 2 (Dropping Multiple Columns)

In [0]:
df.drop('Item_Visibility', 'Item_Type').limit(10).display()

### Dropping Duplicates

In [0]:
df.dropDuplicates().limit(10).display()

#### Scenario - 2

In [0]:
df.dropDuplicates(subset = ['Item_Type']).limit(20).display()

In [0]:
df.distinct().limit(10).display() #cal also be used to drop duplicates 

### Union & Union By Name

#### Preparing DataFrame

In [0]:
data1 = [('1', 'Suryansh'), 
       ('2', 'Aman')]
schema1 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1, schema1)

data2 = [('3', 'Saransh'), 
       ('4', 'Akshay')]
schema2 = 'id String, name String'

df2 = spark.createDataFrame(data2, schema2)

data3 = [('Akash', '5'),
         ('Samar', '6')]
schema3 = 'name STRING, id String'

df3 = spark.createDataFrame(data3, schema3)

In [0]:
df1.display()

In [0]:
df2.display()

#### Union

In [0]:
df1.union(df2).display()

In [0]:
df1.union(df3).display() #its not gonna work if the columns arent in same order in both the table, it will give up messy table

In [0]:
df1.unionByName(df2).unionByName(df3).display()

### String Functions

#### Initcap()

In [0]:
df.select(initcap('Item_Type').alias('Upper_Item_type')).limit(10).display()

#### Lower

In [0]:
df.select(lower('Item_Type')).limit(10).display()

#### Upper

In [0]:
df.select(upper('Item_Type')).limit(10).display()

In [0]:
df.select(col('Item_Type'), col('Item_Weight')).display()

### Date Functions

#### Current Date

In [0]:
df1 = df1.withColumn('curr_date', current_date())
df1.limit(10).display()

#### Date_Add()

In [0]:
df1 = df1.withColumn('week_later', date_add('curr_date', 7)) #js add the number to add the number of days you want to go ahead
df1.display()
df1 = df1.withColumn('week_before', dateadd('curr_date', -7)) # js use - to go back to the number of days you want

#### DateDIFF   

In [0]:
df1.withColumn('datediff', datediff('week_later', 'curr_date')).display()

### Date_Format()

In [0]:
df1.withColumn('week_before', date_format('week_before', 'dd-MM-yyyy')).display()

### Handling Nulls


#### Dropping Nulls

In [0]:
df.dropna('all') #to drop rows which contain nulls in all the column

In [0]:
df.dropna('any') #will drop rows which contain nulls in any of the column

In [0]:
df.dropna(subset = ['Outlet_Size']).limit(10).display() #will check column order type and will drop that row if there is null in that column

### Filling Nulls

In [0]:
df.fillna('NotAvailable').limit(10).display()

In [0]:
df.fillna('Not Availabe', subset = ['Outlet_Size']).limit(10).display()